# 12-phase2-alpha-init-sweep

**neuron Phase 2** — α small-nonzero init sweep (dead block 회피 첫 시도, 옵션 D).

- **Phase 1 의 결정적 발견** (PR #42): head-level granularity 에서도 100% dead. dead 원인은 granularity 가 아닌 `α=0 init 자체의 학습 불능 trade-off`.
- **본 Phase 의 가설**:
  1. **α 작은 nonzero init 으로 dead 회피 가능?** — α_init = 0.01 / 0.03 / 0.1 등에서 학습 후 추가 attn 의 |α| 가 0 에서 벗어나는가?
  2. **function preservation 위반 정도 측정** — α_init 클수록 추가 직후 loss spike 가 더 커짐 (trade-off)
  3. **sweet spot** — dead 회피 + spike 최소화의 균형 α_init 값 존재?
- **데이터**: TinyShakespeare (Phase 1 와 동일)
- **시드**: 42 (단일 시드 — α_init sweep 자체가 4 run, 시드별 비교는 후속)
- **작성일**: 2026-05-25
- **연관**: Issue [#43](https://github.com/EinSofINTEREST/GraphLM/issues/43) / Phase 1 baseline PR [#42](https://github.com/EinSofINTEREST/GraphLM/pull/42)

## 1. 환경 / 시드 / device

In [ ]:
import logging
import sys

import torch
import torch.nn.functional as F

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    iter_random_batches,
    load_tinyshakespeare_text,
)
from graphlm.neuron import NeuronConfig, NeuronGrowingDecoder, add_attn_smooth_start
from graphlm.training.triggers import PlateauTrigger
from graphlm.utils import repo_root, set_seed

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

set_seed(SEED)
logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)

print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)
if str(DEVICE).startswith("cuda"):
    print(f"  device 0      : {torch.cuda.get_device_name(0)}")

## 2. 실험 설정

Phase 1 와 동일 hyperparameter, **α_init 만 sweep**:
- `0.0` — baseline (Phase 1 와 동등, dead 100% 재현 기대)
- `0.01` — 작은 nonzero (가장 신중)
- `0.03` — 중간
- `0.1` — 다소 큼 (spike 위험)

In [ ]:
ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-neuron-phase2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ALPHA_INITS = [0.0, 0.01, 0.03, 0.1]

BATCH_SIZE = 16
BLOCK_SIZE = 128

BACKBONE = dict(hidden_dim=256, n_heads=4, ffn_dim=1024, n_layers=4, n_init_attn=1)
TRAIN = dict(
    lr=3e-4,
    max_steps=1500,
    max_total_attn=8,
    trigger_window=100,
    trigger_epsilon=0.04,
    trigger_cooldown=150,
    trigger_min_history=100,
)
DEAD_THRESHOLD = 0.05  # |α_final| < 0.05 이면 dead
SPIKE_WINDOW = 30  # grow event 전후 30 step 평균 비교

print("ALPHA_INITS =", ALPHA_INITS)
for k, v in {**BACKBONE, **TRAIN}.items():
    print(f"  {k:22s} = {v}")

## 3. 데이터 로드

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
print(f"vocab_size : {tokenizer.vocab_size}, n_tokens : {len(dataset):,}")

## 4. α_init sweep 학습

각 α_init 마다 fresh model + train 1500 step. round-robin attn add. GPU 약 1.5분 × 4 ≈ 6분.

In [ ]:
cfg = NeuronConfig(vocab_size=tokenizer.vocab_size, max_seq_len=BLOCK_SIZE, **BACKBONE)

# {alpha_init: {'losses': [...], 'grow_events': [...], 'final_alphas': [...]}}
runs = {}
for alpha_init in ALPHA_INITS:
    print(f"--- alpha_init = {alpha_init} ---")
    set_seed(SEED)
    model = NeuronGrowingDecoder(cfg).to(DEVICE)
    data_iter = iter_random_batches(
        dataset, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, seed=SEED
    )
    trigger = PlateauTrigger(
        window=TRAIN["trigger_window"],
        epsilon=TRAIN["trigger_epsilon"],
        cooldown=TRAIN["trigger_cooldown"],
        min_history=TRAIN["trigger_min_history"],
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN["lr"])

    losses = []
    grow_events = []
    next_block_to_grow = 0
    V = cfg.vocab_size
    model.train()
    for step in range(1, TRAIN["max_steps"] + 1):
        x, y = next(data_iter)
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

        if trigger.update(loss.item()) and sum(model.n_attn_per_block) < TRAIN["max_total_attn"]:
            target_block = next_block_to_grow
            add_attn_smooth_start(model, target_block, alpha_init=alpha_init)
            block = model.blocks[target_block]
            new_params = (
                list(block.attn_lns[-1].parameters())
                + list(block.attns[-1].parameters())
                + [block.attn_alphas[-1]]
            )
            # lr 명시 — 누락 시 AdamW default lr (1e-3) 적용되어 기존 그룹과 불일치,
            # α_init sweep 간 비교 정확성 훼손. weight_decay 는 동일 default (0.01) 라 생략 가능.
            optimizer.add_param_group({"params": new_params, "lr": TRAIN["lr"]})
            grow_events.append((step, target_block, sum(model.n_attn_per_block)))
            next_block_to_grow = (next_block_to_grow + 1) % cfg.n_layers

    # 결과 수집
    final_alphas = [
        (bi, ai, alpha_param.item())
        for bi, block in enumerate(model.blocks)
        for ai, alpha_param in enumerate(block.attn_alphas)
    ]
    runs[alpha_init] = {
        "losses": losses,
        "grow_events": grow_events,
        "final_alphas": final_alphas,
        "n_attn_per_block": list(model.n_attn_per_block),
    }
    print(
        f"  done: total_attn={sum(model.n_attn_per_block)}, grow_events={len(grow_events)}, "
        f"final_loss≈{losses[-1]:.4f}"
    )

## 5. α_init 별 결과 표 — dead 비율 + spike 정도 + final loss

In [ ]:
import statistics

print(
    f"{'α_init':>7}  {'#grow':>6}  {'#added':>7}  {'#dead':>6}  {'dead%':>6}  "
    f"{'mean|α_add|':>11}  {'max_spike':>10}  {'final_loss':>11}"
)
print("-" * 90)
summary = []
for alpha_init, r in runs.items():
    # added attn 만 (block.attn_idx >= n_init_attn)
    added = [(bi, ai, a) for bi, ai, a in r["final_alphas"] if ai >= BACKBONE["n_init_attn"]]
    added_abs = [abs(a) for _, _, a in added]
    n_dead = sum(1 for v in added_abs if v < DEAD_THRESHOLD)
    dead_pct = n_dead / len(added_abs) if added_abs else 0.0
    mean_add = statistics.mean(added_abs) if added_abs else 0.0

    # spike 계산 — 각 grow event 전후 30 step 평균 비교
    losses = r["losses"]
    spikes = []
    for step, _b, _t in r["grow_events"]:
        pre = losses[max(0, step - SPIKE_WINDOW) : step]
        post = losses[step : step + SPIKE_WINDOW]
        if pre and post:
            spikes.append(sum(post) / len(post) - sum(pre) / len(pre))
    max_spike = max(spikes) if spikes else 0.0

    n_last = min(100, len(losses))
    final_loss = sum(losses[-n_last:]) / n_last if n_last > 0 else 0.0

    summary.append((alpha_init, dead_pct, mean_add, max_spike, final_loss))
    print(
        f"{alpha_init:>7.2f}  {len(r['grow_events']):>6}  {len(added_abs):>7}  {n_dead:>6}  "
        f"{dead_pct:>6.1%}  {mean_add:>11.4f}  {max_spike:>+10.4f}  {final_loss:>11.4f}"
    )
print()
print("Phase 1 비교 (α_init=0.0): dead 100%, max_spike ≈ 0")

## 6. trade-off 시각화 — dead 비율 vs max_spike

In [ ]:
import matplotlib.pyplot as plt

alpha_vals = [s[0] for s in summary]
dead_pcts = [s[1] for s in summary]
max_spikes = [s[3] for s in summary]
final_losses = [s[4] for s in summary]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# (a) α_init vs dead% + spike
color1 = "steelblue"
ax1.set_xlabel("α_init")
ax1.set_ylabel("dead %", color=color1)
ax1.plot(alpha_vals, dead_pcts, "o-", color=color1, label="dead %")
ax1.tick_params(axis="y", labelcolor=color1)
ax1.set_ylim(-0.05, 1.05)
ax1.grid(alpha=0.3)
ax1b = ax1.twinx()
color2 = "orangered"
ax1b.set_ylabel("max spike (Δloss)", color=color2)
ax1b.plot(alpha_vals, max_spikes, "s-", color=color2, label="max spike")
ax1b.tick_params(axis="y", labelcolor=color2)
ax1.set_title("trade-off: α_init → dead 회피 vs spike")

# (b) α_init vs final loss
ax2.plot(alpha_vals, final_losses, "o-", color="steelblue")
ax2.set_xlabel("α_init")
ax2.set_ylabel("final loss (last 100 avg)")
ax2.set_title("α_init vs final loss")
ax2.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(OUT_DIR / "alpha_init_sweep.png", dpi=120)
plt.show()

## 7. α_init 별 attention α 분포 (grouped bar)

In [ ]:
fig, axes = plt.subplots(len(ALPHA_INITS), 1, figsize=(11, 2.2 * len(ALPHA_INITS)), sharex=True)
if len(ALPHA_INITS) == 1:
    axes = [axes]
for ax, alpha_init in zip(axes, ALPHA_INITS, strict=True):
    r = runs[alpha_init]
    x_pos = []
    heights = []
    colors = []
    labels = []
    for x, (bi, ai, a) in enumerate(r["final_alphas"]):
        x_pos.append(x)
        heights.append(abs(a))
        is_init = ai < BACKBONE["n_init_attn"]
        colors.append("steelblue" if is_init else "orange")
        labels.append(f"b{bi}.a{ai}")
    ax.bar(x_pos, heights, color=colors, alpha=0.85)
    ax.axhline(DEAD_THRESHOLD, color="red", linestyle="--", lw=1, alpha=0.5)
    ax.set_ylabel("|α|")
    ax.set_title(f"α_init = {alpha_init}", fontsize=10)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, rotation=45, fontsize=7)
    ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(OUT_DIR / "alpha_distribution_by_init.png", dpi=120)
plt.show()

## 결과 요약 / 권장 α_init / 다음 phase

이 노트북에서 확인할 것:
- §5 의 요약 표 — α_init 별 dead %, max_spike, final_loss
- §6 의 trade-off 곡선 — dead 회피와 spike 의 sweet spot
- §7 의 α 분포 — 추가 attn 의 막대 (주황) 가 dead threshold (빨간 선) 보다 얼마나 위로 올라오는지

**판정 기준** (가설별):
1. **dead 회피**: α_init ↑ 시 dead % 가 100% → X% 로 감소하면 옵션 D 의 유효성 입증
2. **spike 허용 범위**: max_spike < 0.3 정도면 학습에 큰 영향 없음 (Phase 1 기존 baseline 의 spike 도 ±0.05 수준)
3. **sweet spot**: dead % 낮음 + spike 작음 + final_loss 동등/우위인 α_init

**예상 시나리오**:
- **A. dead 비율 그래도 100%** → α_init=0.1 까지도 dead 면 옵션 D 의 한계. 옵션 B (GradMax SVD init) / C (local trigger) 로 진행.
- **B. dead 회피 + spike 큼** → α_init=0.1 에서 dead 0% 인데 spike 크면 작은 α_init 으로 후속 sweep (0.005, 0.02 등) 정밀 탐색.
- **C. sweet spot 존재** → 권장 α_init 으로 Phase 1 의 baseline 재실험 + 시드 다양화 (Phase 3).

**Phase 3 후보 (sweet spot 존재 시)**:
- 시드 invariance 재검증 (옵션 D 의 결과가 robust 한지)
- 다양한 trigger 와 결합 (옵션 C 와 hybrid)
- 옵션 A (connection-level α) 로 확장